# Test of the final Gold audit (`09_gold_final_audit`)

This notebook tests `09_gold_final_audit` with small hand-made tables in the schema `github_lakehouse.test_gold_final`. It builds the six Gold tables there with the six real Gold notebooks, and runs the audit with `%run` and `test_schema = test_gold_final`. So **production tables are never written**. The last check section proves that.

It has the same kind of sections as the tests of the six Gold notebooks. Three things are different, because the audit is different:

- The audit writes no Gold table. Its result is its audit row and its list of checks.
- The test **breaks the Gold tables on purpose**, in one cell, so that every one of the 33 checks must fail. Then it sets the tables back with `RESTORE TABLE`.
- The test adds new Silver rows, one of them late, and runs the audit before and after the Gold tables are rebuilt, as the Job will.

**Before you run it:** `08_gold_audit_table` and `07_source_hour_manifest` must have run once, and the nine Silver tables must exist. The test copies the definitions of the tables. It does not read their rows.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Six cells are EXPECTED to fail.** The text before each of them says so. "Run all" stops at such a cell. Then click the cell **after** it and choose "Run all below". It does no harm if one of these six cells runs more than once.
- **The result of the whole test is the cell "Result of the whole test" near the end.** It prints `TEST PASSED` only when all 13 sections ran and every check passed. If a section failed or was left out, it stops with `TEST FAILED` and names the section.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: empty tables give `NO_INPUT`, no checks, and an audit row with zeros |
| The audit row | T4 to T22: every column of the audit row, for every run |
| Exact numbers | T8, T10: the Silver rows, the repository-days and the events of each Gold table, worked out by hand |
| G5 adds up | T7, T8: without a successful run of G5 only `G5_DOES_NOT_ADD_UP_TO_SILVER` fails. T10: each of the seven reasons of G5 is added. T19 to T22: the newest successful run of G5 is used |
| All dates and one date | T10, T12: all dates compare the whole tables. One date compares only its days, also a day before it that got a late row. T19 to T22: a day that only a late release event brings into a source date |
| A source date without rows | T13, T14: `NO_INPUT` |
| The audit writes no Gold table | T14: the six Gold tables have the same versions as before the audit runs |
| Every check can fail | T15, T16: one cell breaks the six tables. All 33 checks fail, each with the number worked out by hand |
| Recovery | T17, T18: the tables are set back, and the audit succeeds again |
| A Gold table that is not rebuilt | T19 to T22: a new push and two new release events fail the audit until G3, G5 and G6 are rebuilt for their source date |
| Bad parameter | T23 to T25: an invalid `source_date`, `test_schema` or `run_id` stops the notebook in its parameter step. T26: nothing was written |
| Production safety | T26: no production table changed |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean.

`CREATE TABLE ... LIKE` makes an empty table with the same columns as another table. The test tables are empty copies of the real tables, with the same names, only in the test schema:

- the nine Silver tables,
- the source-hour manifest, because G5 reads it,
- the audit table.

The six Gold test tables are created by the six Gold notebooks, in their first run.

In [0]:
TEST_SCHEMA = "test_gold_final"

TEST = f"github_lakehouse.{TEST_SCHEMA}"
PROD_SILVER = "github_lakehouse.silver"

# The nine Silver tables.
SILVER_TABLE_NAMES = [
    "push_events",
    "pull_request_events",
    "issue_events",
    "issue_comment_events",
    "pull_request_review_events",
    "pull_request_review_comment_events",
    "watch_events",
    "fork_events",
    "release_events",
]

# The Silver test tables.
TEST_PUSH = TEST + ".push_events"
TEST_PULL_REQUEST = TEST + ".pull_request_events"
TEST_ISSUE = TEST + ".issue_events"
TEST_ISSUE_COMMENT = TEST + ".issue_comment_events"
TEST_REVIEW = TEST + ".pull_request_review_events"
TEST_REVIEW_COMMENT = TEST + ".pull_request_review_comment_events"
TEST_WATCH = TEST + ".watch_events"
TEST_FORK = TEST + ".fork_events"
TEST_RELEASE = TEST + ".release_events"

# The Gold test tables. The Gold notebooks create them.
TEST_G1 = TEST + ".repository_delivery_metrics"
TEST_G2 = TEST + ".repository_issue_flow"
TEST_G3 = TEST + ".repository_contributor_concentration"
TEST_G4 = TEST + ".repository_collaboration_metrics"
TEST_G5 = TEST + ".release_activity_shift"
TEST_G6 = TEST + ".repository_activity_profile"

TEST_GOLD_TABLES = {
    "G1": TEST_G1,
    "G2": TEST_G2,
    "G3": TEST_G3,
    "G4": TEST_G4,
    "G5": TEST_G5,
    "G6": TEST_G6,
}

TEST_MANIFEST = TEST + ".source_hour_manifest"
TEST_AUDIT = TEST + ".gold_pipeline_runs"

PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_MANIFEST = "github_lakehouse.audit.source_hour_manifest"

# The production tables whose versions are compared at the start and at the end of the test.
PROD_TABLES = [
    "github_lakehouse.audit.gold_pipeline_runs",
    "github_lakehouse.audit.source_hour_manifest",
    "github_lakehouse.gold.repository_delivery_metrics",
    "github_lakehouse.gold.repository_issue_flow",
    "github_lakehouse.gold.repository_contributor_concentration",
    "github_lakehouse.gold.repository_collaboration_metrics",
    "github_lakehouse.gold.release_activity_shift",
    "github_lakehouse.gold.repository_activity_profile",
    "github_lakehouse.gold_staging.repository_delivery_metrics",
    "github_lakehouse.gold_staging.repository_issue_flow",
    "github_lakehouse.gold_staging.repository_contributor_concentration",
    "github_lakehouse.gold_staging.repository_collaboration_metrics",
    "github_lakehouse.gold_staging.release_activity_shift",
    "github_lakehouse.gold_staging.repository_activity_profile",
]

# The columns of the watch and fork rows. The other columns of the Silver tables stay NULL.
SEVEN_COLUMNS = "(event_id, event_timestamp, event_date, actor_id, repo_id, repo_name, _source_date)"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Every check section writes its number of failed checks into this dictionary.
# The cell "Result of the whole test" reads it at the end.
section_results = {}

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

if not spark.catalog.tableExists(PROD_MANIFEST):
    raise RuntimeError(f"{PROD_MANIFEST} does not exist. Run 07_source_hour_manifest first.")

for table_name in SILVER_TABLE_NAMES:
    if not spark.catalog.tableExists(PROD_SILVER + "." + table_name):
        raise RuntimeError(
            f"{PROD_SILVER}.{table_name} does not exist. Run the Silver notebook of this table first."
        )

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

for table_name in SILVER_TABLE_NAMES:
    spark.sql(f"CREATE TABLE {TEST}.{table_name} LIKE {PROD_SILVER}.{table_name}")

spark.sql(f"CREATE TABLE {TEST_MANIFEST} LIKE {PROD_MANIFEST}")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The test runs all six Gold notebooks, so it checks 14 tables: the audit table, the manifest, the six Gold tables and their six staging tables. A table that does not exist yet gets the value `-1`.

In [0]:
prod_versions_before = {}

for table_name in PROD_TABLES:
    if spark.catalog.tableExists(table_name):
        prod_versions_before[table_name] = spark.sql(
            f"DESCRIBE HISTORY {table_name} LIMIT 1"
        ).collect()[0]["version"]
    else:
        prod_versions_before[table_name] = -1

    print(table_name, ":", prod_versions_before[table_name])

### A short summary of every audit row of the final audit

An audit row holds two maps: the Silver rows read per table, and the quality numbers. The checks below compare them as text. This view writes each of them into one readable line, for example:

`push=6 pull_requests=4 issues=4 issue_comments=3 reviews=2 review_comments=1 watches=5 forks=1 releases=10`

`checks=33 failed=0 g1=3/4 g2=3/4 g3=5/20 g4=3/6 g6=7/36 g5=3+7 all_releases=10`

In the second line, `g1=3/4` means: G1 has 3 rows with 4 events on the days that were compared. `g5=3+7` means: G5 has 3 rows, and its last successful run left out 7 release events. `none` means that there was no successful run of G5. `all_releases` is the number of all release events in Silver.

The view is not a copy. It always reads the audit table as it is at that moment.

In [0]:
final_audit_summary = spark.sql(f"""
    SELECT
        run_id,
        concat(
            'push=',              CAST(source_rows_read['push_events'] AS STRING),
            ' pull_requests=',    CAST(source_rows_read['pull_request_events'] AS STRING),
            ' issues=',           CAST(source_rows_read['issue_events'] AS STRING),
            ' issue_comments=',   CAST(source_rows_read['issue_comment_events'] AS STRING),
            ' reviews=',          CAST(source_rows_read['pull_request_review_events'] AS STRING),
            ' review_comments=',  CAST(source_rows_read['pull_request_review_comment_events'] AS STRING),
            ' watches=',          CAST(source_rows_read['watch_events'] AS STRING),
            ' forks=',            CAST(source_rows_read['fork_events'] AS STRING),
            ' releases=',         CAST(source_rows_read['release_events'] AS STRING)
        ) AS source_summary,
        concat(
            'checks=',  CAST(quality_metrics['checks_run'] AS STRING),
            ' failed=', CAST(quality_metrics['checks_failed'] AS STRING),
            ' g1=', CAST(quality_metrics['g1_rows'] AS STRING), '/', CAST(quality_metrics['g1_events'] AS STRING),
            ' g2=', CAST(quality_metrics['g2_rows'] AS STRING), '/', CAST(quality_metrics['g2_events'] AS STRING),
            ' g3=', CAST(quality_metrics['g3_rows'] AS STRING), '/', CAST(quality_metrics['g3_events'] AS STRING),
            ' g4=', CAST(quality_metrics['g4_rows'] AS STRING), '/', CAST(quality_metrics['g4_events'] AS STRING),
            ' g6=', CAST(quality_metrics['g6_rows'] AS STRING), '/', CAST(quality_metrics['g6_events'] AS STRING),
            ' g5=', CAST(quality_metrics['g5_rows'] AS STRING), '+',
                    coalesce(CAST(quality_metrics['g5_left_out_release_events'] AS STRING), 'none'),
            ' all_releases=', CAST(quality_metrics['silver_release_events_all_days'] AS STRING)
        ) AS metrics_summary
    FROM {TEST_AUDIT}
    WHERE pipeline_name = 'gold_final_audit'
""")
final_audit_summary.createOrReplaceTempView("final_audit_summary")

## T3. Run on the empty tables

First the six Gold notebooks, so that the six Gold test tables exist, then the audit. Every Silver table is empty, so every run must end as `NO_INPUT`.

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_final" $source_date="" $run_id="test-g1-empty"

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_final" $source_date="" $run_id="test-g2-empty"

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_final" $source_date="" $run_id="test-g3-empty"

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_final" $source_date="" $run_id="test-g4-empty"

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_final" $source_date="" $run_id="test-g5-empty"

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_final" $source_date="" $run_id="test-g6-empty"

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="" $run_id="test-audit-empty"

## T4. Checks: no input

There is no day to compare, so the audit runs no check and ends as `NO_INPUT`. It still writes its audit row.

The first 16 checks compare the table names the notebook used with the names of the test tables.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test push table' AS check_name,
               '{PUSH_TABLE}' AS actual, '{TEST_PUSH}' AS expected
        UNION ALL
        SELECT '02 the notebook read the test pull request table', '{PULL_REQUEST_TABLE}', '{TEST_PULL_REQUEST}'
        UNION ALL
        SELECT '03 the notebook read the test issue table', '{ISSUE_TABLE}', '{TEST_ISSUE}'
        UNION ALL
        SELECT '04 the notebook read the test issue comment table', '{ISSUE_COMMENT_TABLE}', '{TEST_ISSUE_COMMENT}'
        UNION ALL
        SELECT '05 the notebook read the test review table', '{REVIEW_TABLE}', '{TEST_REVIEW}'
        UNION ALL
        SELECT '06 the notebook read the test review comment table', '{REVIEW_COMMENT_TABLE}', '{TEST_REVIEW_COMMENT}'
        UNION ALL
        SELECT '07 the notebook read the test watch table', '{WATCH_TABLE}', '{TEST_WATCH}'
        UNION ALL
        SELECT '08 the notebook read the test fork table', '{FORK_TABLE}', '{TEST_FORK}'
        UNION ALL
        SELECT '09 the notebook read the test release table', '{RELEASE_TABLE}', '{TEST_RELEASE}'
        UNION ALL
        SELECT '10 the notebook read the test G1 table', '{G1_TABLE}', '{TEST_G1}'
        UNION ALL
        SELECT '11 the notebook read the test G2 table', '{G2_TABLE}', '{TEST_G2}'
        UNION ALL
        SELECT '12 the notebook read the test G3 table', '{G3_TABLE}', '{TEST_G3}'
        UNION ALL
        SELECT '13 the notebook read the test G4 table', '{G4_TABLE}', '{TEST_G4}'
        UNION ALL
        SELECT '14 the notebook read the test G5 table', '{G5_TABLE}', '{TEST_G5}'
        UNION ALL
        SELECT '15 the notebook read the test G6 table', '{G6_TABLE}', '{TEST_G6}'
        UNION ALL
        SELECT '16 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '17 rows in the six Gold test tables',
               CAST(
                   (SELECT COUNT(*) FROM {TEST_G1}) + (SELECT COUNT(*) FROM {TEST_G2})
                 + (SELECT COUNT(*) FROM {TEST_G3}) + (SELECT COUNT(*) FROM {TEST_G4})
                 + (SELECT COUNT(*) FROM {TEST_G5}) + (SELECT COUNT(*) FROM {TEST_G6})
               AS STRING),
               '0'
        UNION ALL
        SELECT '18 the six Gold runs ended as NO_INPUT',
               CAST(count_if(pipeline_name <> 'gold_final_audit' AND status = 'NO_INPUT') AS STRING), '6'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '19 audit rows of the final audit', CAST(count_if(pipeline_name = 'gold_final_audit') AS STRING), '1'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '20 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '21 audit pipeline_name', MAX(pipeline_name), 'gold_final_audit'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '22 audit gold_table: no table written', MAX(gold_table), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '23 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '24 audit source_date', CAST(MAX(source_date) AS STRING), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '25 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '26 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '27 audit failed_checks', MAX(failed_checks), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '28 audit error_message', MAX(error_message), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '29 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '30 audit gold_table_version', CAST(MAX(gold_table_version) AS STRING), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '31 audit Silver rows read', MAX(source_summary),
               'push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 review_comments=0 watches=0 forks=0 releases=0'
        FROM final_audit_summary WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '32 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '33 audit quality_metrics', MAX(metrics_summary),
               'checks=0 failed=0 g1=0/0 g2=0/0 g3=0/0 g4=0/0 g6=0/0 g5=0+none all_releases=0'
        FROM final_audit_summary WHERE run_id = 'test-audit-empty'
        UNION ALL
        SELECT '34 checks run by the notebook', '{len(check_results)}', '0'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T4"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made rows

### T5.1 The manifest

G5 needs the manifest: a release is compared only when every hour of its windows is complete. `range(168)` gives the numbers 0 to 167, and `timestampadd(HOUR, id, ...)` adds that many hours to 1 February 00:00. So the query gives the 168 hours of the week, all complete.

In [0]:
spark.sql(f"""
    INSERT INTO {TEST_MANIFEST} (source_hour_utc, source_date, is_complete)
    SELECT
        timestampadd(HOUR, id, TIMESTAMP'2024-02-01 00:00:00')                   AS source_hour_utc,
        CAST(timestampadd(HOUR, id, TIMESTAMP'2024-02-01 00:00:00') AS DATE)     AS source_date,
        true                                                                     AS is_complete
    FROM range(168)
""")

display(spark.sql(f"""
    SELECT
        COUNT(*)                        AS hours,
        count_if(is_complete = true)    AS complete_hours
    FROM {TEST_MANIFEST}
"""))

### T5.2 The Silver rows

37 Silver rows in the nine tables. Event IDs start with `77000000` and exist only once in the nine tables together. The numbers in brackets are the actor IDs.

| Repository | Day | push | pull request | issue | issue comment | review | inline comment | watch | fork | release |
|---|---|---|---|---|---|---|---|---|---|---|
| 100 | 2 Feb | 3 (1, 1, 2) | 2 (1, 1) | 1 (3) | 2 (3, 4) | 1 (2) | 1 (2) | 1 (5) | 1 (none) | |
| 100 | 3 Feb | 2 (1, 1) | 1 (2) | | | | | | | 1 (1) |
| 200 | 2 Feb | | 1 (6), late | | | | | | | |
| 200 | 3 Feb | | | 2 (6, 7) | 1 (6) | | | 2 (8, 9) | | |
| 300 | 4 Feb | | | | | | | 1 (10) | | 8 (10) |
| 400 | 4 Feb | 1 (11) | | 1 (11) | | 1 (12) | | | | 1 (11), a draft |
| 600 | 3 Feb | | | | | | | 1 (13) | | |
| 100 | no date | 1 (1) | | | | | | | | |

- Every row has its own day as its source date, with two exceptions, both with the source date 3 February. The **late** pull request of repository 200 happened on 2 February at 23:59:58 and came in a file of 3 February. The push without a date cannot be placed on a day.
- The two issue comments of repository 100 are on an issue and in a pull request conversation. The comment of repository 200 is on an issue.
- Three releases become G5 rows: repository 100 at 3 February 12:00, and repository 300 at 4 February 06:00 and 18:00.
- Seven release events are left out by G5, one for each of its seven reasons: the draft of repository 400 (`DRAFT_OR_UNKNOWN`), and six events of repository 300. These are an event that is not `published` (`NOT_PUBLISHED`), one without a published time (`MISSING_PUBLISHED_AT`), one whose prerelease flag is unknown (`PRERELEASE_UNKNOWN`), one without a release ID (`MISSING_RELEASE_ID`), a second event of release 5002 (`REPEATED_RELEASE_EVENT`), and a release published on 31 January, whose window is outside the week (`WINDOW_NOT_COVERED`).
- Repository 300 has only a watch and a release, and repository 600 only a watch. So they have a G6 row, but no row in G1 to G4. The test uses this in T15.

**What the Gold tables must hold, worked out by hand:**

| Table | Kinds | Rows (repository-days) | Events | |
|---|---|---|---|---|
| G1 | pull request | 3 | 4 | 100 on 2 and 3 Feb, 200 on 2 Feb |
| G2 | issue | 3 | 4 | 100 on 2 Feb, 200 on 3 Feb, 400 on 4 Feb |
| G3 | the six engineering kinds | 5 | 20 | 10 + 3 + 1 + 3 + 3 |
| G4 | issue comment, review, inline comment | 3 | 6 | 4 + 1 + 1 |
| G6 | all nine | 7 | 36 | 12 + 4 + 1 + 5 + 9 + 4 + 1 |
| G5 | release | 3 | | Plus 7 release events left out: 3 + 7 = 10 release events |

The undated push is in none of them. It is counted in the audit row as `missing_event_date`.

The people per repository-day, to check G6 ≥ G3 ≥ G4:

| Repository-day | G6 | G3 | G4 |
|---|---|---|---|
| 100, 2 Feb | 5: 1, 2, 3, 4, 5 | 4: 1, 2, 3, 4 | 3: 2, 3, 4 |
| 100, 3 Feb | 2: 1, 2 | 2: 1, 2 | no row |
| 200, 2 Feb | 1: 6 | 1: 6 | no row |
| 200, 3 Feb | 4: 6, 7, 8, 9 | 2: 6, 7 | 1: 6 |
| 300, 4 Feb | 1: 10 | no row | no row |
| 400, 4 Feb | 2: 11, 12 | 2: 11, 12 | 1: 12 |
| 600, 3 Feb | 1: 13 | no row | no row |

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     push_commit_count, push_distinct_commit_count, _source_date)
VALUES
    ('77000000001', TIMESTAMP'2024-02-02 09:00:00', DATE'2024-02-02', 1,  100, 'org/alpha', 1, 1, DATE'2024-02-02'),
    ('77000000002', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', 1,  100, 'org/alpha', 2, 2, DATE'2024-02-02'),
    ('77000000003', TIMESTAMP'2024-02-02 11:00:00', DATE'2024-02-02', 2,  100, 'org/alpha', 1, 1, DATE'2024-02-02'),
    ('77000000004', TIMESTAMP'2024-02-03 09:00:00', DATE'2024-02-03', 1,  100, 'org/alpha', 1, 1, DATE'2024-02-03'),
    ('77000000005', TIMESTAMP'2024-02-03 10:00:00', DATE'2024-02-03', 1,  100, 'org/alpha', 3, 3, DATE'2024-02-03'),
    ('77000000006', TIMESTAMP'2024-02-04 08:00:00', DATE'2024-02-04', 11, 400, 'org/delta', 1, 1, DATE'2024-02-04'),

    -- A push without an event time and an event date
    ('77000000007', NULL,                           NULL,             1,  100, 'org/alpha', 1, 1, DATE'2024-02-03')
""")

spark.sql("INSERT INTO " + TEST_PULL_REQUEST + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     pr_action, pr_id, pr_number, pr_created_at, pr_closed_at, pr_merged_at, pr_merged,
     pr_commits, pr_additions, pr_deletions, pr_changed_files, _source_date)
VALUES
    ('77000000011', TIMESTAMP'2024-02-02 09:30:00', DATE'2024-02-02', 1, 100, 'org/alpha', 'opened', 1001, 1,
     TIMESTAMP'2024-02-02 09:30:00', NULL, NULL, false, 1, 10, 0, 1, DATE'2024-02-02'),
    ('77000000012', TIMESTAMP'2024-02-02 12:00:00', DATE'2024-02-02', 1, 100, 'org/alpha', 'closed', 1001, 1,
     TIMESTAMP'2024-02-02 09:30:00', TIMESTAMP'2024-02-02 12:00:00', TIMESTAMP'2024-02-02 12:00:00', true, 1, 10, 0, 1, DATE'2024-02-02'),
    ('77000000013', TIMESTAMP'2024-02-03 11:00:00', DATE'2024-02-03', 2, 100, 'org/alpha', 'opened', 1002, 2,
     TIMESTAMP'2024-02-03 11:00:00', NULL, NULL, false, 2, 20, 5, 2, DATE'2024-02-03'),

    -- The late pull request: it happened on 2 February and came with the source date 3 February
    ('77000000014', TIMESTAMP'2024-02-02 23:59:58', DATE'2024-02-02', 6, 200, 'org/beta',  'opened', 2001, 1,
     TIMESTAMP'2024-02-02 23:59:58', NULL, NULL, false, 1, 3, 0, 1, DATE'2024-02-03')
""")

spark.sql("INSERT INTO " + TEST_ISSUE + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     issue_action, issue_id, issue_number, issue_created_at, issue_closed_at, _source_date)
VALUES
    ('77000000021', TIMESTAMP'2024-02-02 13:00:00', DATE'2024-02-02', 3,  100, 'org/alpha', 'opened', 3001, 3, TIMESTAMP'2024-02-02 13:00:00', NULL, DATE'2024-02-02'),
    ('77000000022', TIMESTAMP'2024-02-03 08:00:00', DATE'2024-02-03', 6,  200, 'org/beta',  'opened', 4001, 2, TIMESTAMP'2024-02-03 08:00:00', NULL, DATE'2024-02-03'),
    ('77000000023', TIMESTAMP'2024-02-03 09:00:00', DATE'2024-02-03', 7,  200, 'org/beta',  'opened', 4002, 3, TIMESTAMP'2024-02-03 09:00:00', NULL, DATE'2024-02-03'),
    ('77000000024', TIMESTAMP'2024-02-04 09:30:00', DATE'2024-02-04', 11, 400, 'org/delta', 'opened', 6001, 1, TIMESTAMP'2024-02-04 09:30:00', NULL, DATE'2024-02-04')
""")

spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     comment_action, comment_target_type, _source_date)
VALUES
    ('77000000031', TIMESTAMP'2024-02-02 14:00:00', DATE'2024-02-02', 3, 100, 'org/alpha', 'created', 'issue',        DATE'2024-02-02'),
    ('77000000032', TIMESTAMP'2024-02-02 15:00:00', DATE'2024-02-02', 4, 100, 'org/alpha', 'created', 'pull_request', DATE'2024-02-02'),
    ('77000000033', TIMESTAMP'2024-02-03 10:00:00', DATE'2024-02-03', 6, 200, 'org/beta',  'created', 'issue',        DATE'2024-02-03')
""")

spark.sql("INSERT INTO " + TEST_REVIEW + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     review_action, review_state, _source_date)
VALUES
    ('77000000041', TIMESTAMP'2024-02-02 11:30:00', DATE'2024-02-02', 2,  100, 'org/alpha', 'created', 'approved',  DATE'2024-02-02'),
    ('77000000042', TIMESTAMP'2024-02-04 10:00:00', DATE'2024-02-04', 12, 400, 'org/delta', 'created', 'commented', DATE'2024-02-04')
""")

spark.sql("INSERT INTO " + TEST_REVIEW_COMMENT + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     comment_action, _source_date)
VALUES
    ('77000000051', TIMESTAMP'2024-02-02 11:40:00', DATE'2024-02-02', 2, 100, 'org/alpha', 'created', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
VALUES
    ('77000000061', TIMESTAMP'2024-02-02 16:00:00', DATE'2024-02-02', 5,  100, 'org/alpha', DATE'2024-02-02'),
    ('77000000062', TIMESTAMP'2024-02-03 12:00:00', DATE'2024-02-03', 8,  200, 'org/beta',  DATE'2024-02-03'),
    ('77000000063', TIMESTAMP'2024-02-03 13:00:00', DATE'2024-02-03', 9,  200, 'org/beta',  DATE'2024-02-03'),
    ('77000000064', TIMESTAMP'2024-02-03 14:00:00', DATE'2024-02-03', 13, 600, 'org/zeta',  DATE'2024-02-03'),
    ('77000000065', TIMESTAMP'2024-02-04 07:00:00', DATE'2024-02-04', 10, 300, 'org/gamma', DATE'2024-02-04')
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + SEVEN_COLUMNS + """
VALUES
    ('77000000071', TIMESTAMP'2024-02-02 17:00:00', DATE'2024-02-02', NULL, 100, 'org/alpha', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_RELEASE + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     release_action, release_id, release_tag, release_published_at, release_draft, release_prerelease, _source_date)
VALUES
    ('77000000081', TIMESTAMP'2024-02-03 12:00:05', DATE'2024-02-03', 1,  100, 'org/alpha', 'published', 5001, 'v1.0',
     TIMESTAMP'2024-02-03 12:00:00', false, false, DATE'2024-02-03'),
    ('77000000082', TIMESTAMP'2024-02-04 06:00:04', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', 5002, 'v2.0',
     TIMESTAMP'2024-02-04 06:00:00', false, false, DATE'2024-02-04'),
    ('77000000084', TIMESTAMP'2024-02-04 18:00:03', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', 5004, 'v2.1',
     TIMESTAMP'2024-02-04 18:00:00', false, false, DATE'2024-02-04'),

    -- Left out by G5, one event for each reason
    -- DRAFT_OR_UNKNOWN: a draft
    ('77000000083', TIMESTAMP'2024-02-04 11:00:00', DATE'2024-02-04', 11, 400, 'org/delta', 'published', 5003, 'v0.1',
     NULL, true, false, DATE'2024-02-04'),
    -- REPEATED_RELEASE_EVENT: a second, later event of release 5002
    ('77000000085', TIMESTAMP'2024-02-04 20:00:02', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', 5002, 'v2.0',
     TIMESTAMP'2024-02-04 20:00:00', false, false, DATE'2024-02-04'),
    -- NOT_PUBLISHED: the action is not published
    ('77000000086', TIMESTAMP'2024-02-04 08:00:00', DATE'2024-02-04', 10, 300, 'org/gamma', 'created',   5005, 'v2.2',
     TIMESTAMP'2024-02-04 08:00:00', false, false, DATE'2024-02-04'),
    -- MISSING_PUBLISHED_AT: no published time
    ('77000000087', TIMESTAMP'2024-02-04 08:30:00', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', 5006, 'v2.3',
     NULL, false, false, DATE'2024-02-04'),
    -- PRERELEASE_UNKNOWN: the prerelease flag is unknown
    ('77000000088', TIMESTAMP'2024-02-04 09:00:05', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', 5007, 'v2.4',
     TIMESTAMP'2024-02-04 09:00:00', false, NULL, DATE'2024-02-04'),
    -- MISSING_RELEASE_ID: no release ID
    ('77000000089', TIMESTAMP'2024-02-04 10:00:05', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', NULL, 'v2.5',
     TIMESTAMP'2024-02-04 10:00:00', false, false, DATE'2024-02-04'),
    -- WINDOW_NOT_COVERED: published on 31 January, so its window has hours before the week
    ('77000000090', TIMESTAMP'2024-02-04 13:00:00', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', 5008, 'v1.9',
     TIMESTAMP'2024-01-31 12:00:00', false, false, DATE'2024-02-04')
""")

display(spark.sql(f"""
    SELECT 'push' AS silver_table, COUNT(*) AS test_rows FROM {TEST_PUSH}
    UNION ALL SELECT 'pull_request',   COUNT(*) FROM {TEST_PULL_REQUEST}
    UNION ALL SELECT 'issue',          COUNT(*) FROM {TEST_ISSUE}
    UNION ALL SELECT 'issue_comment',  COUNT(*) FROM {TEST_ISSUE_COMMENT}
    UNION ALL SELECT 'review',         COUNT(*) FROM {TEST_REVIEW}
    UNION ALL SELECT 'review_comment', COUNT(*) FROM {TEST_REVIEW_COMMENT}
    UNION ALL SELECT 'watch',          COUNT(*) FROM {TEST_WATCH}
    UNION ALL SELECT 'fork',           COUNT(*) FROM {TEST_FORK}
    UNION ALL SELECT 'release',        COUNT(*) FROM {TEST_RELEASE}
"""))

## T6. Build five Gold tables, but not G5

G1, G2, G3, G4 and G6 are built for all dates. G5 is left out on purpose: its last run is the `NO_INPUT` run of T3, so it has no successful run yet.

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_final" $source_date="" $run_id="test-g1-run1"

In [0]:
%run ../02_repository_issue_flow $test_schema="test_gold_final" $source_date="" $run_id="test-g2-run1"

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_final" $source_date="" $run_id="test-g3-run1"

In [0]:
%run ../04_repository_collaboration_metrics $test_schema="test_gold_final" $source_date="" $run_id="test-g4-run1"

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_final" $source_date="" $run_id="test-g6-run1"

## T7. Run 1 of the audit: G5 has no successful run

G5 has no row and no successful run. So the audit cannot add up the release events, and only `G5_DOES_NOT_ADD_UP_TO_SILVER` must fail. The other 32 checks must pass: G5 is empty, so it has no key twice and no repository outside G6.

**The next cell is EXPECTED to fail** with `RuntimeError: gold_final_audit FAILED: Checks failed: G5_DOES_NOT_ADD_UP_TO_SILVER`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="" $run_id="test-audit-run1"

## T8. Checks of run 1

The audit row is written before the notebook stops, so it is there. `check_results` is the list of checks of the notebook. `%run` runs the other notebook inside this one, so its variables can be read here, also after it stopped with an error. The test makes a small table of it, `audit_checks`.

All dates are compared: 2, 3 and 4 February. The numbers are those of the tables in T5.2.

In [0]:
spark.createDataFrame(
    check_results,
    "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
).createOrReplaceTempView("audit_checks")

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit rows of run 1 that are not FAILED' AS check_name,
               CAST(count_if(status <> 'FAILED') AS STRING) AS actual, '0' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '02 audit status', MAX(status), 'FAILED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '03 audit failed_checks', MAX(failed_checks), 'G5_DOES_NOT_ADD_UP_TO_SILVER'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '05 audit error_message', MAX(error_message), 'Checks failed: G5_DOES_NOT_ADD_UP_TO_SILVER'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '06 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '07 audit affected_dates: all days with rows', MAX(CAST(affected_dates AS STRING)), '[2024-02-02, 2024-02-03, 2024-02-04]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '08 audit Silver rows read', MAX(source_summary),
               'push=6 pull_requests=4 issues=4 issue_comments=3 reviews=2 review_comments=1 watches=5 forks=1 releases=10'
        FROM final_audit_summary WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '09 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '10 audit quality_metrics', MAX(metrics_summary),
               'checks=33 failed=1 g1=3/4 g2=3/4 g3=5/20 g4=3/6 g6=7/36 g5=0+none all_releases=10'
        FROM final_audit_summary WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '11 audit gold_table', MAX(gold_table), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '12 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run1'
        UNION ALL
        SELECT '13 checks run', CAST(COUNT(*) AS STRING), '33' FROM audit_checks
        UNION ALL
        SELECT '14 checks failed', CAST(count_if(NOT passed) AS STRING), '1' FROM audit_checks
        UNION ALL
        SELECT '15 the G5 check: found (no successful run)', CAST(MAX(found) AS STRING), NULL
        FROM audit_checks WHERE check_name = 'G5_DOES_NOT_ADD_UP_TO_SILVER'
        UNION ALL
        SELECT '16 the G5 check: expected', CAST(MAX(expected) AS STRING), '10'
        FROM audit_checks WHERE check_name = 'G5_DOES_NOT_ADD_UP_TO_SILVER'
        UNION ALL
        SELECT '17 the five Gold runs of T6 succeeded',
               CAST(count_if(run_id IN ('test-g1-run1', 'test-g2-run1', 'test-g3-run1', 'test-g4-run1', 'test-g6-run1')
                             AND status = 'SUCCEEDED') AS STRING), '5'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T8"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T8: {failed_count} check(s) failed. See the table above.")
print("T8: ALL CHECKS PASSED")

## T9. Build G5, then run 2 of the audit

G5 is built now. Then the Delta versions of the six Gold test tables are written down. They are used twice: T14 proves that the audit runs did not change the tables, and T15 and T17 set the tables back to these versions.

Run 2 compares all dates. Every check must pass.

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_final" $source_date="" $run_id="test-g5-run1"

In [0]:
gold_versions_after_build = {}

for table_label in TEST_GOLD_TABLES:
    gold_versions_after_build[table_label] = spark.sql(
        f"DESCRIBE HISTORY {TEST_GOLD_TABLES[table_label]} LIMIT 1"
    ).collect()[0]["version"]
    print(table_label, "version:", gold_versions_after_build[table_label])

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="" $run_id="test-audit-run2"

## T10. Checks of run 2

All 33 checks pass. The numbers are the same as in run 1, and G5 now adds up: 3 rows + 7 release events left out = 10 release events. The run of G5 left out one release event for each of its seven reasons, so each reason is part of the sum.

In [0]:
spark.createDataFrame(
    check_results,
    "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
).createOrReplaceTempView("audit_checks")

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '03 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '04 audit error_message', MAX(error_message), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '05 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '06 audit source_date', CAST(MAX(source_date) AS STRING), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '07 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-02, 2024-02-03, 2024-02-04]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '08 audit Silver rows read', MAX(source_summary),
               'push=6 pull_requests=4 issues=4 issue_comments=3 reviews=2 review_comments=1 watches=5 forks=1 releases=10'
        FROM final_audit_summary WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '09 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '10 audit quality_metrics', MAX(metrics_summary),
               'checks=33 failed=0 g1=3/4 g2=3/4 g3=5/20 g4=3/6 g6=7/36 g5=3+7 all_releases=10'
        FROM final_audit_summary WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '11 audit gold_table', MAX(gold_table), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '12 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '13 audit gold_table_version', CAST(MAX(gold_table_version) AS STRING), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '14 audit rows of run 2', CAST(COUNT(*) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run2'
        UNION ALL
        SELECT '15 checks run', CAST(COUNT(*) AS STRING), '33' FROM audit_checks
        UNION ALL
        SELECT '16 checks failed', CAST(count_if(NOT passed) AS STRING), '0' FROM audit_checks
        UNION ALL
        SELECT '17 the G5 run: status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-g5-run1'
        UNION ALL
        SELECT '18 the G5 run: rows published', CAST(MAX(gold_rows_published) AS STRING), '3'
        FROM {TEST_AUDIT} WHERE run_id = 'test-g5-run1'
        UNION ALL
        SELECT '19 the G5 run: one release event left out for each reason',
               MAX(concat(
                   'not_published=',         CAST(quality_metrics['NOT_PUBLISHED'] AS STRING),
                   ' draft=',                CAST(quality_metrics['DRAFT_OR_UNKNOWN'] AS STRING),
                   ' no_published_at=',      CAST(quality_metrics['MISSING_PUBLISHED_AT'] AS STRING),
                   ' prerelease_unknown=',   CAST(quality_metrics['PRERELEASE_UNKNOWN'] AS STRING),
                   ' no_release_id=',        CAST(quality_metrics['MISSING_RELEASE_ID'] AS STRING),
                   ' repeated=',             CAST(quality_metrics['REPEATED_RELEASE_EVENT'] AS STRING),
                   ' not_covered=',          CAST(quality_metrics['WINDOW_NOT_COVERED'] AS STRING)
               )),
               'not_published=1 draft=1 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=1 not_covered=1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-g5-run1'
        UNION ALL
        SELECT '20 the G5 check: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '10 and 10'
        FROM audit_checks WHERE check_name = 'G5_DOES_NOT_ADD_UP_TO_SILVER'
        UNION ALL
        SELECT '21 the G6 rows check: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '7 and 7'
        FROM audit_checks WHERE check_name = 'G6_ROWS_DIFFER_FROM_SILVER'
        UNION ALL
        SELECT '22 the G3 events check: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '20 and 20'
        FROM audit_checks WHERE check_name = 'G3_EVENTS_DIFFER_FROM_SILVER'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T10"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T10: {failed_count} check(s) failed. See the table above.")
print("T10: ALL CHECKS PASSED")

## T11. Run 3: one source date

`source_date = 2024-02-03`. The Silver rows of that source date have the event dates 2 and 3 February: the late pull request of repository 200 happened on 2 February. So both days are compared, and 4 February is not.

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="2024-02-03" $run_id="test-audit-run3"

## T12. Checks of run 3

Worked out by hand, only on 2 and 3 February:

| Table | Rows | Events |
|---|---|---|
| G1 | 3 | 4 |
| G2 | 2: 100 on 2 Feb, 200 on 3 Feb | 3 |
| G3 | 4 | 17 = 10 + 3 + 1 + 3 |
| G4 | 2 | 5 = 4 + 1 |
| G6 | 5 | 23 = 12 + 4 + 1 + 5 + 1 |

Silver on these days: 5 pushes, 4 pull requests, 3 issues, 3 issue comments, 1 review, 1 inline comment, 4 watches, 1 fork and 1 release. The push without a date has the source date 3 February, so it is counted as `missing_event_date`.

G5 is always compared as a whole: 3 + 7 = 10, all release events.

In [0]:
spark.createDataFrame(
    check_results,
    "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
).createOrReplaceTempView("audit_checks")

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-03'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '04 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-02, 2024-02-03]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary),
               'push=5 pull_requests=4 issues=3 issue_comments=3 reviews=1 review_comments=1 watches=4 forks=1 releases=1'
        FROM final_audit_summary WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(metrics_summary),
               'checks=33 failed=0 g1=3/4 g2=2/3 g3=4/17 g4=2/5 g6=5/23 g5=3+7 all_releases=10'
        FROM final_audit_summary WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '08 audit failed_checks', MAX(failed_checks), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run3'
        UNION ALL
        SELECT '09 checks run', CAST(COUNT(*) AS STRING), '33' FROM audit_checks
        UNION ALL
        SELECT '10 checks failed', CAST(count_if(NOT passed) AS STRING), '0' FROM audit_checks
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T12"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T12: {failed_count} check(s) failed. See the table above.")
print("T12: ALL CHECKS PASSED")

## T13. Run 4: a source date without rows

No Silver row has the source date 1 March 2024. So there is no day to compare, and the run must end as `NO_INPUT`.

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="2024-03-01" $run_id="test-audit-run4"

## T14. Checks of run 4, and the audit wrote no Gold table

The last six checks compare the versions of the six Gold test tables with the versions of T9. Runs 2, 3 and 4 came after T9, so the versions must be the same: the audit only reads the Gold tables.

In [0]:
gold_versions_now = {}

for table_label in TEST_GOLD_TABLES:
    gold_versions_now[table_label] = spark.sql(
        f"DESCRIBE HISTORY {TEST_GOLD_TABLES[table_label]} LIMIT 1"
    ).collect()[0]["version"]

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'NO_INPUT' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-03-01'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '04 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary),
               'push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 review_comments=0 watches=0 forks=0 releases=0'
        FROM final_audit_summary WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(metrics_summary),
               'checks=0 failed=0 g1=0/0 g2=0/0 g3=0/0 g4=0/0 g6=0/0 g5=0+none all_releases=0'
        FROM final_audit_summary WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '08 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run4'
        UNION ALL
        SELECT '09 checks run by the notebook', '{len(check_results)}', '0'
        UNION ALL
        SELECT '10 G1 version unchanged by the audit', '{gold_versions_now["G1"]}', '{gold_versions_after_build["G1"]}'
        UNION ALL
        SELECT '11 G2 version unchanged by the audit', '{gold_versions_now["G2"]}', '{gold_versions_after_build["G2"]}'
        UNION ALL
        SELECT '12 G3 version unchanged by the audit', '{gold_versions_now["G3"]}', '{gold_versions_after_build["G3"]}'
        UNION ALL
        SELECT '13 G4 version unchanged by the audit', '{gold_versions_now["G4"]}', '{gold_versions_after_build["G4"]}'
        UNION ALL
        SELECT '14 G5 version unchanged by the audit', '{gold_versions_now["G5"]}', '{gold_versions_after_build["G5"]}'
        UNION ALL
        SELECT '15 G6 version unchanged by the audit', '{gold_versions_now["G6"]}', '{gold_versions_after_build["G6"]}'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T14"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T14: {failed_count} check(s) failed. See the table above.")
print("T14: ALL CHECKS PASSED")

## T15. Break the six Gold tables, then run 5

This cell changes the Gold test tables on purpose, so that **every one of the 33 checks must fail**. Silver is not changed.

It first sets the six tables back to their versions of T9 with `RESTORE TABLE`. So the cell gives the same tables when it runs twice.

There are two kinds of changes. "Copy" adds a second, equal row with `INSERT INTO ... SELECT`. "Set" changes one row with `UPDATE`.

| | Table | Change | The checks it breaks |
|---|---|---|---|
| B1 | G1 | Copy the row of 100 on 3 Feb | G1 key twice, G1 rows and events |
| B2 | G1 | Set `repo_id` NULL on the row of 200 on 2 Feb | G1 key NULL, G6 against G1, a G1 row not in G6 |
| B3 | G2 | Set `issue_event_count` from 2 to 3 on the row of 200 on 3 Feb | G2 events, G6 against G2 |
| B4 | G2 | Copy the row of 100 on 2 Feb | G2 key twice, G2 rows and events |
| B5 | G2 | Set `activity_date` NULL on the row of 400 | G2 key NULL, G6 against G2, a G2 row not in G6 |
| B6 | G3 | Copy the row of 400 | G3 key twice, G3 rows and events |
| B7 | G3 | Set `activity_date` NULL on the row of 100 on 3 Feb | G3 key NULL, G6 against G3, people G6 ≥ G3, a G3 row not in G6 |
| B8 | G4 | Set `unique_collaborating_actors` from 3 to 99 on the row of 100 | People G3 ≥ G4 |
| B9 | G4 | Copy the row of 200 | G4 key twice, G4 rows and events |
| B10 | G4 | Set `activity_date` NULL on the row of 400 | G4 key NULL, G6 against G4, people G3 ≥ G4, a G4 row not in G6 |
| B11 | G6 | On the row of 100 on 2 Feb, move one comment from `pr_conversation_comment_events` to `unclassified_conversation_comment_events` | The comment split |
| B12 | G5 | Copy the row of repository 100 | G5 key twice, G5 does not add up |
| B13 | G5 | Set `release_anchor_key` NULL on the row of release 5002 of repository 300 | G5 key NULL |
| B14 | G6 | Set `unique_known_actors` from 5 to 1 on the row of 100 on 2 Feb | People G6 ≥ G3 |
| B15 | G6 | Copy the row of 600 | G6 key twice, G6 rows, events and kinds |
| B16 | G6 | Set `repo_id` NULL on the row of 300 | G6 key NULL, a G5 repository not in G6 |

B11 moves a number from one column to another, so the total of the row stays right. Only the comment split check can see it. B15 copies a row that has no partner in G1 to G4, so it breaks only the checks of G6 itself.

In [0]:
for table_label in TEST_GOLD_TABLES:
    spark.sql(
        f"RESTORE TABLE {TEST_GOLD_TABLES[table_label]} TO VERSION AS OF {gold_versions_after_build[table_label]}"
    )

# B1. G1: copy the row of repository 100 on 3 February.
spark.sql(f"""
    INSERT INTO {TEST_G1}
    SELECT * FROM {TEST_G1}
    WHERE repo_id = 100 AND activity_date = DATE'2024-02-03'
""")

# B2. G1: repo_id NULL on the row of repository 200 on 2 February.
spark.sql(f"""
    UPDATE {TEST_G1}
    SET repo_id = NULL
    WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
""")

# B3. G2: one issue event too many on the row of repository 200 on 3 February.
spark.sql(f"""
    UPDATE {TEST_G2}
    SET issue_event_count = issue_event_count + 1
    WHERE repo_id = 200 AND activity_date = DATE'2024-02-03'
""")

# B4. G2: copy the row of repository 100 on 2 February.
spark.sql(f"""
    INSERT INTO {TEST_G2}
    SELECT * FROM {TEST_G2}
    WHERE repo_id = 100 AND activity_date = DATE'2024-02-02'
""")

# B5. G2: activity_date NULL on the row of repository 400.
spark.sql(f"""
    UPDATE {TEST_G2}
    SET activity_date = NULL
    WHERE repo_id = 400 AND activity_date = DATE'2024-02-04'
""")

# B6. G3: copy the row of repository 400.
spark.sql(f"""
    INSERT INTO {TEST_G3}
    SELECT * FROM {TEST_G3}
    WHERE repo_id = 400 AND activity_date = DATE'2024-02-04'
""")

# B7. G3: activity_date NULL on the row of repository 100 on 3 February.
spark.sql(f"""
    UPDATE {TEST_G3}
    SET activity_date = NULL
    WHERE repo_id = 100 AND activity_date = DATE'2024-02-03'
""")

# B8. G4: 99 people on the row of repository 100.
spark.sql(f"""
    UPDATE {TEST_G4}
    SET unique_collaborating_actors = 99
    WHERE repo_id = 100 AND activity_date = DATE'2024-02-02'
""")

# B9. G4: copy the row of repository 200.
spark.sql(f"""
    INSERT INTO {TEST_G4}
    SELECT * FROM {TEST_G4}
    WHERE repo_id = 200 AND activity_date = DATE'2024-02-03'
""")

# B10. G4: activity_date NULL on the row of repository 400.
spark.sql(f"""
    UPDATE {TEST_G4}
    SET activity_date = NULL
    WHERE repo_id = 400 AND activity_date = DATE'2024-02-04'
""")

# B11. G6: one comment moved to the wrong column on the row of repository 100 on 2 February.
spark.sql(f"""
    UPDATE {TEST_G6}
    SET pr_conversation_comment_events = pr_conversation_comment_events - 1,
        unclassified_conversation_comment_events = unclassified_conversation_comment_events + 1
    WHERE repo_id = 100 AND activity_date = DATE'2024-02-02'
""")

# B12. G5: copy the row of repository 100.
spark.sql(f"""
    INSERT INTO {TEST_G5}
    SELECT * FROM {TEST_G5}
    WHERE repo_id = 100
""")

# B13. G5: release_anchor_key NULL on the row of release 5002 of repository 300.
spark.sql(f"""
    UPDATE {TEST_G5}
    SET release_anchor_key = NULL
    WHERE repo_id = 300 AND release_anchor_key = '5002'
""")

# B14. G6: 1 person instead of 5 on the row of repository 100 on 2 February.
spark.sql(f"""
    UPDATE {TEST_G6}
    SET unique_known_actors = 1
    WHERE repo_id = 100 AND activity_date = DATE'2024-02-02'
""")

# B15. G6: copy the row of repository 600.
spark.sql(f"""
    INSERT INTO {TEST_G6}
    SELECT * FROM {TEST_G6}
    WHERE repo_id = 600 AND activity_date = DATE'2024-02-03'
""")

# B16. G6: repo_id NULL on the row of repository 300.
spark.sql(f"""
    UPDATE {TEST_G6}
    SET repo_id = NULL
    WHERE repo_id = 300 AND activity_date = DATE'2024-02-04'
""")

display(spark.sql(f"""
    SELECT 'G1' AS gold_table, COUNT(*) AS test_rows FROM {TEST_G1}
    UNION ALL SELECT 'G2', COUNT(*) FROM {TEST_G2}
    UNION ALL SELECT 'G3', COUNT(*) FROM {TEST_G3}
    UNION ALL SELECT 'G4', COUNT(*) FROM {TEST_G4}
    UNION ALL SELECT 'G5', COUNT(*) FROM {TEST_G5}
    UNION ALL SELECT 'G6', COUNT(*) FROM {TEST_G6}
"""))

### Run 5

**The next cell is EXPECTED to fail** with `RuntimeError: gold_final_audit FAILED: Checks failed: G1_KEY_NOT_UNIQUE; G1_KEY_IS_NULL; ...` and all 33 check names. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="" $run_id="test-audit-run5"

## T16. Checks of run 5: every check failed, with the number worked out by hand

The table below has one line per check, in the order the notebook runs them: the number the check must find, and the number it expects.

**Keys.** Each table has one key twice and one key with NULL, so every key check finds 1.

**Gold against Silver.** Silver did not change, so the expected numbers are those of T5.2. A copied row adds one row and its events: B1 adds 1 pull request to G1, B4 1 issue to G2, B6 the 3 events of repository 400 to G3, B9 1 comment to G4, and B15 1 watch to G6. B3 adds 1 more issue to G2. In G6 only the watches are wrong now, so one kind differs. G5 has 4 rows now, plus 7 release events left out: 11 instead of 10.

**Gold against Gold.** A key set to NULL has no partner in the other table any more. Then **two** rows differ: the changed row, and the row on the other side that lost its partner. So:

| Check | Found | Why |
|---|---|---|
| G6 against G1 | 2 | B2: the G1 row with NULL, and the G6 row of 200 on 2 Feb |
| G6 against G2 | 3 | B3: one row. B5: the G2 row with NULL, and the G6 row of 400 |
| G6 against G3 | 2 | B7: the G3 row with NULL, and the G6 row of 100 on 3 Feb |
| G6 against G4 | 2 | B10: the G4 row with NULL, and the G6 row of 400 |
| The comment split | 1 | B11. The row of B10 has no comments, so its split is 0 on both sides |
| People G6 ≥ G3 | 2 | B14: 1 < 4. B7: the G3 row with NULL finds no G6 row, so 0 < 2 |
| People G3 ≥ G4 | 2 | B8: 4 < 99. B10: the G4 row with NULL finds no G3 row, so 0 < 1 |
| A row of G1 to G4 not in G6 | 4 | The four rows with a NULL key, from B2, B5, B7 and B10 |
| A G5 repository not in G6 | 1 | B16: repository 300. It has two G5 rows, but the check counts repositories |

A copied row finds its partner twice, with the same numbers, so B1, B4, B6 and B9 do not change the checks of Gold against Gold.

In [0]:
EXPECTED_RUN5 = [
    # check name,                          found, expected
    ("G1_KEY_NOT_UNIQUE",                      1,  0),
    ("G1_KEY_IS_NULL",                         1,  0),
    ("G2_KEY_NOT_UNIQUE",                      1,  0),
    ("G2_KEY_IS_NULL",                         1,  0),
    ("G3_KEY_NOT_UNIQUE",                      1,  0),
    ("G3_KEY_IS_NULL",                         1,  0),
    ("G4_KEY_NOT_UNIQUE",                      1,  0),
    ("G4_KEY_IS_NULL",                         1,  0),
    ("G5_KEY_NOT_UNIQUE",                      1,  0),
    ("G5_KEY_IS_NULL",                         1,  0),
    ("G6_KEY_NOT_UNIQUE",                      1,  0),
    ("G6_KEY_IS_NULL",                         1,  0),
    ("G1_ROWS_DIFFER_FROM_SILVER",             4,  3),
    ("G1_EVENTS_DIFFER_FROM_SILVER",           5,  4),
    ("G2_ROWS_DIFFER_FROM_SILVER",             4,  3),
    ("G2_EVENTS_DIFFER_FROM_SILVER",           6,  4),
    ("G3_ROWS_DIFFER_FROM_SILVER",             6,  5),
    ("G3_EVENTS_DIFFER_FROM_SILVER",          23, 20),
    ("G4_ROWS_DIFFER_FROM_SILVER",             4,  3),
    ("G4_EVENTS_DIFFER_FROM_SILVER",           7,  6),
    ("G6_ROWS_DIFFER_FROM_SILVER",             8,  7),
    ("G6_EVENTS_DIFFER_FROM_SILVER",          37, 36),
    ("G6_KIND_DIFFERS_FROM_SILVER",            1,  0),
    ("G5_DOES_NOT_ADD_UP_TO_SILVER",          11, 10),
    ("G6_PULL_REQUESTS_DIFFER_FROM_G1",        2,  0),
    ("G6_ISSUES_DIFFER_FROM_G2",               3,  0),
    ("G6_ENGINEERING_DIFFERS_FROM_G3",         2,  0),
    ("G6_COLLABORATION_DIFFERS_FROM_G4",       2,  0),
    ("G6_COMMENT_SPLIT_DIFFERS_FROM_G4",       1,  0),
    ("ACTORS_G6_BELOW_G3",                     2,  0),
    ("ACTORS_G3_BELOW_G4",                     2,  0),
    ("G1_TO_G4_ROW_NOT_IN_G6",                 4,  0),
    ("G5_REPOSITORY_NOT_IN_G6",                1,  0),
]

# The audit row must name all 33 checks, in this order.
expected_names = []
for check_name, found, expected in EXPECTED_RUN5:
    expected_names.append(check_name)
expected_failed_checks = "; ".join(expected_names)

spark.createDataFrame(
    check_results,
    "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
).createOrReplaceTempView("audit_checks")

spark.createDataFrame(
    EXPECTED_RUN5,
    "check_name STRING, found BIGINT, expected BIGINT",
).createOrReplaceTempView("expected_checks")

# One line per check: what the notebook found and expected, next to the numbers worked out by hand.
check_lines = spark.sql("""
    SELECT
        e.check_name,
        concat(coalesce(CAST(a.found AS STRING), 'NULL'), ' / ', coalesce(CAST(a.expected AS STRING), 'NULL'),
               ' / ', coalesce(CAST(a.passed AS STRING), 'NULL')) AS actual,
        concat(CAST(e.found AS STRING), ' / ', CAST(e.expected AS STRING), ' / false') AS expected
    FROM expected_checks e
    LEFT JOIN audit_checks a
        ON a.check_name = e.check_name
""")
check_lines.createOrReplaceTempView("check_lines")

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT check_name, actual, expected FROM check_lines
        UNION ALL
        SELECT 'checks run', CAST(COUNT(*) AS STRING), '33' FROM audit_checks
        UNION ALL
        SELECT 'audit rows of run 5 that are not FAILED', CAST(count_if(status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run5'
        UNION ALL
        SELECT 'audit failed_checks: all 33, in order', MAX(failed_checks), '{expected_failed_checks}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run5'
        UNION ALL
        SELECT 'audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run5'
        UNION ALL
        SELECT 'audit quality_metrics', MAX(metrics_summary),
               'checks=33 failed=33 g1=4/5 g2=4/6 g3=6/23 g4=4/7 g6=8/37 g5=4+7 all_releases=10'
        FROM final_audit_summary WHERE run_id = 'test-audit-run5'
    )
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T16"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T16: {failed_count} check(s) failed. See the table above.")
print("T16: ALL CHECKS PASSED")

## T17. Set the tables back, then run 6

`RESTORE TABLE ... TO VERSION AS OF` sets a Delta table back to an older version. This is how a broken Gold table is repaired without building it again. The six tables go back to their versions of T9. Then every check must pass again.

In [0]:
for table_label in TEST_GOLD_TABLES:
    spark.sql(
        f"RESTORE TABLE {TEST_GOLD_TABLES[table_label]} TO VERSION AS OF {gold_versions_after_build[table_label]}"
    )
    print(table_label, "is back at version", gold_versions_after_build[table_label])

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="" $run_id="test-audit-run6"

## T18. Checks of run 6

The same numbers as run 2.

In [0]:
spark.createDataFrame(
    check_results,
    "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
).createOrReplaceTempView("audit_checks")

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run6'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run6'
        UNION ALL
        SELECT '03 audit quality_metrics', MAX(metrics_summary),
               'checks=33 failed=0 g1=3/4 g2=3/4 g3=5/20 g4=3/6 g6=7/36 g5=3+7 all_releases=10'
        FROM final_audit_summary WHERE run_id = 'test-audit-run6'
        UNION ALL
        SELECT '04 checks run', CAST(COUNT(*) AS STRING), '33' FROM audit_checks
        UNION ALL
        SELECT '05 checks failed', CAST(count_if(NOT passed) AS STRING), '0' FROM audit_checks
        UNION ALL
        SELECT '06 G1 rows after the restore', CAST(COUNT(*) AS STRING), '3' FROM {TEST_G1}
        UNION ALL
        SELECT '07 G5 rows after the restore', CAST(COUNT(*) AS STRING), '3' FROM {TEST_G5}
        UNION ALL
        SELECT '08 G6 rows after the restore', CAST(COUNT(*) AS STRING), '7' FROM {TEST_G6}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T18"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Three new Silver rows, then run 7 before the Gold tables are rebuilt

Three rows arrive with the source date 4 February: a push of a new repository, 500, a third event of release 5002 of repository 300, and a **late** release event of repository 100. The late event happened on 3 February at 23:59:59 and came in a file of 4 February. Its action is `created`, so G5 will leave it out as `NOT_PUBLISHED`. Silver has the three rows now, but no Gold table was built again.

Run 7 compares the days of the source date 4 February, as the Job will after the Gold tasks of that date. These are 3 and 4 February now, and only the late release event brings 3 February into this source date. So if the audit forgot the release table when it looks for the days, it would compare only 4 February, and T20 would fail.

The push is one of the six kinds of G3 and one of the nine kinds of G6. The two release events are of the nine kinds of G6, and G5 has not counted them yet. So exactly these six checks must fail, on 3 and 4 February together:

| Check | Found | Expected |
|---|---|---|
| `G3_ROWS_DIFFER_FROM_SILVER` | 3: repositories 100 and 200 on 3 February, 400 on 4 February | 4: also 500 on 4 February |
| `G3_EVENTS_DIFFER_FROM_SILVER` | 9 | 10 |
| `G6_ROWS_DIFFER_FROM_SILVER` | 5: repositories 100, 200 and 600 on 3 February, 300 and 400 on 4 February | 6 |
| `G6_EVENTS_DIFFER_FROM_SILVER` | 23 | 26 |
| `G6_KIND_DIFFERS_FROM_SILVER` | 2: 3 pushes instead of 4, and 10 releases instead of 12 | 0 |
| `G5_DOES_NOT_ADD_UP_TO_SILVER` | 10: 3 rows and 7 left out, from the run of T9 | 12 |

The other 27 checks pass. G1, G2 and G4 do not read pushes or releases, and G1 to G4 still agree with G6 on every row they have.

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     push_commit_count, push_distinct_commit_count, _source_date)
VALUES
    ('77000000091', TIMESTAMP'2024-02-04 15:00:00', DATE'2024-02-04', 14, 500, 'org/epsilon', 1, 1, DATE'2024-02-04')
""")

spark.sql("INSERT INTO " + TEST_RELEASE + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name,
     release_action, release_id, release_tag, release_published_at, release_draft, release_prerelease, _source_date)
VALUES
    ('77000000092', TIMESTAMP'2024-02-04 22:00:01', DATE'2024-02-04', 10, 300, 'org/gamma', 'published', 5002, 'v2.0',
     TIMESTAMP'2024-02-04 22:00:00', false, false, DATE'2024-02-04'),
    -- The late release event: it happened on 3 February and came with the source date 4 February
    ('77000000093', TIMESTAMP'2024-02-03 23:59:59', DATE'2024-02-03', 1,  100, 'org/alpha', 'created',   5009, 'v1.1',
     TIMESTAMP'2024-02-03 23:59:59', false, false, DATE'2024-02-04')
""")

print("Pushes in Silver now   :", spark.table(TEST_PUSH).count())
print("Releases in Silver now :", spark.table(TEST_RELEASE).count())

### Run 7

**The next cell is EXPECTED to fail** with `RuntimeError: gold_final_audit FAILED: Checks failed: G3_ROWS_DIFFER_FROM_SILVER; G3_EVENTS_DIFFER_FROM_SILVER; G6_ROWS_DIFFER_FROM_SILVER; G6_EVENTS_DIFFER_FROM_SILVER; G6_KIND_DIFFERS_FROM_SILVER; G5_DOES_NOT_ADD_UP_TO_SILVER`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="2024-02-04" $run_id="test-audit-run7"

## T20. Checks of run 7

Silver on 3 and 4 February: 4 pushes, 1 pull request event, 3 issue events, 1 issue comment, 1 review, 4 watches and 12 release events.

In [0]:
spark.createDataFrame(
    check_results,
    "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
).createOrReplaceTempView("audit_checks")

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit rows of run 7 that are not FAILED' AS check_name,
               CAST(count_if(status <> 'FAILED') AS STRING) AS actual, '0' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run7'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks),
               'G3_ROWS_DIFFER_FROM_SILVER; G3_EVENTS_DIFFER_FROM_SILVER; G6_ROWS_DIFFER_FROM_SILVER; G6_EVENTS_DIFFER_FROM_SILVER; G6_KIND_DIFFERS_FROM_SILVER; G5_DOES_NOT_ADD_UP_TO_SILVER'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run7'
        UNION ALL
        SELECT '03 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run7'
        UNION ALL
        SELECT '04 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-03, 2024-02-04]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run7'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary),
               'push=4 pull_requests=1 issues=3 issue_comments=1 reviews=1 review_comments=0 watches=4 forks=0 releases=12'
        FROM final_audit_summary WHERE run_id = 'test-audit-run7'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run7'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(metrics_summary),
               'checks=33 failed=6 g1=1/1 g2=2/3 g3=3/9 g4=2/2 g6=5/23 g5=3+7 all_releases=12'
        FROM final_audit_summary WHERE run_id = 'test-audit-run7'
        UNION ALL
        SELECT '08 checks run', CAST(COUNT(*) AS STRING), '33' FROM audit_checks
        UNION ALL
        SELECT '09 checks failed', CAST(count_if(NOT passed) AS STRING), '6' FROM audit_checks
        UNION ALL
        SELECT '10 G3 rows: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '3 and 4'
        FROM audit_checks WHERE check_name = 'G3_ROWS_DIFFER_FROM_SILVER'
        UNION ALL
        SELECT '11 G3 events: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '9 and 10'
        FROM audit_checks WHERE check_name = 'G3_EVENTS_DIFFER_FROM_SILVER'
        UNION ALL
        SELECT '12 G6 rows: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '5 and 6'
        FROM audit_checks WHERE check_name = 'G6_ROWS_DIFFER_FROM_SILVER'
        UNION ALL
        SELECT '13 G6 events: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '23 and 26'
        FROM audit_checks WHERE check_name = 'G6_EVENTS_DIFFER_FROM_SILVER'
        UNION ALL
        SELECT '14 G6 kinds: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '2 and 0'
        FROM audit_checks WHERE check_name = 'G6_KIND_DIFFERS_FROM_SILVER'
        UNION ALL
        SELECT '15 G5: found and expected', MAX(concat(CAST(found AS STRING), ' and ', CAST(expected AS STRING))), '10 and 12'
        FROM audit_checks WHERE check_name = 'G5_DOES_NOT_ADD_UP_TO_SILVER'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T20"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T20: {failed_count} check(s) failed. See the table above.")
print("T20: ALL CHECKS PASSED")

## T21. Rebuild G3, G5 and G6 for 4 February, then run 8

The three tables that read pushes or releases are built again for the source date 4 February. G1, G2 and G4 read neither, so they stay as they are. G5 is always rebuilt as a whole. Then the audit runs again for the same source date and must succeed.

The audit takes the reasons of G5 from its **newest** successful run, the one of this section. The run of T9 has two release events less.

In [0]:
%run ../03_repository_contributor_concentration $test_schema="test_gold_final" $source_date="2024-02-04" $run_id="test-g3-run2"

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_final" $source_date="2024-02-04" $run_id="test-g5-run2"

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_final" $source_date="2024-02-04" $run_id="test-g6-run2"

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="2024-02-04" $run_id="test-audit-run8"

## T22. Checks of run 8

On 3 and 4 February together, G3 has 4 rows with 10 events now: 2 rows on each day. G6 was rebuilt for both days and has 6 rows with 26 events. G5 still has 3 rows, and now 9 release events left out: the third event of release 5002 is a second `REPEATED_RELEASE_EVENT`, and the late event is a second `NOT_PUBLISHED`. 3 + 9 = 12.

In [0]:
spark.createDataFrame(
    check_results,
    "check_name STRING, found BIGINT, expected BIGINT, passed BOOLEAN",
).createOrReplaceTempView("audit_checks")

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run8'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), NULL
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run8'
        UNION ALL
        SELECT '03 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-03, 2024-02-04]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-audit-run8'
        UNION ALL
        SELECT '04 audit Silver rows read', MAX(source_summary),
               'push=4 pull_requests=1 issues=3 issue_comments=1 reviews=1 review_comments=0 watches=4 forks=0 releases=12'
        FROM final_audit_summary WHERE run_id = 'test-audit-run8'
        UNION ALL
        SELECT '05 audit quality_metrics', MAX(metrics_summary),
               'checks=33 failed=0 g1=1/1 g2=2/3 g3=4/10 g4=2/2 g6=6/26 g5=3+9 all_releases=12'
        FROM final_audit_summary WHERE run_id = 'test-audit-run8'
        UNION ALL
        SELECT '06 checks run', CAST(COUNT(*) AS STRING), '33' FROM audit_checks
        UNION ALL
        SELECT '07 checks failed', CAST(count_if(NOT passed) AS STRING), '0' FROM audit_checks
        UNION ALL
        SELECT '08 the three Gold runs of T21 succeeded',
               CAST(count_if(run_id IN ('test-g3-run2', 'test-g5-run2', 'test-g6-run2') AND status = 'SUCCEEDED') AS STRING), '3'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '09 the new G5 run: repeated release events', CAST(MAX(quality_metrics['REPEATED_RELEASE_EVENT']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-g5-run2'
        UNION ALL
        SELECT '10 the new G5 run: release events that are not published', CAST(MAX(quality_metrics['NOT_PUBLISHED']) AS STRING), '2'
        FROM {TEST_AUDIT} WHERE run_id = 'test-g5-run2'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T22"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T22: {failed_count} check(s) failed. See the table above.")
print("T22: ALL CHECKS PASSED")

## T23. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="2024-2-3" $run_id="test-bad-date"

### T23 check: where did it stop?

`%run` runs the other notebook inside this one, so the variables of that notebook can be read here. They show how far the stopped run came:

- `source_date` holds the bad value, so step 1.2 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4, where it reads the run ID. It stopped before it touched a table.

In [0]:
print("source_date:", source_date)
print("run_id     :", run_id)

if source_date != "2024-2-3":
    raise AssertionError("T23: the run with the bad source_date has not run. Run the cell above first.")

if run_id == "test-bad-date":
    raise AssertionError("T23: the notebook did not stop at the bad source_date. It went on to step 1.4.")

section_results["T23"] = 0
print("T23: the notebook stopped at the bad source_date, as expected")

## T24. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../09_gold_final_audit $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema"

### T24 check: where did it stop?

- `test_schema` holds the bad value, so step 1.3 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if test_schema != "TEST_WRONG":
    raise AssertionError("T24: the run with the bad test_schema has not run. Run the cell above first.")

if run_id == "test-bad-schema":
    raise AssertionError("T24: the notebook did not stop at the bad test_schema. It went on to step 1.4.")

section_results["T24"] = 0
print("T24: the notebook stopped at the bad test_schema, as expected")

## T25. Bad parameter: `run_id`

The run ID goes into the audit row, so it may contain only letters, digits, `-` and `_`. `bad.run.id` contains dots.

**The next cell is EXPECTED to fail** with `ValueError: Invalid run_id 'bad.run.id'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../09_gold_final_audit $test_schema="test_gold_final" $source_date="" $run_id="bad.run.id"

### T25 check: did it run?

- `run_id` holds the bad value, so the notebook came to step 1.4.
- That it stopped there, and wrote nothing, is checked in T26: this run has no audit row.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if run_id != "bad.run.id":
    raise AssertionError("T25: the run with the bad run_id has not run. Run the cell above first.")

section_results["T25"] = 0
print("T25: the notebook came to the run_id step. T26 checks that it wrote nothing.")

## T26. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so they have no audit row.
- The final audit ran 9 times: 4 succeeded, 3 failed on purpose and 2 had no input. Every run wrote exactly one audit row. Runs 1, 5 and 7 are left out of that check, because their cells may have run more than once.
- No audit row of the final audit names a Gold table, a Gold row or a table version.
- The six Gold notebooks ran 15 times: 6 without input in T3, 5 in T6, G5 in T9, and G3, G5 and G6 in T21.
- The versions of the 14 production tables are read again and compared with the versions of T2.

In [0]:
prod_versions_after = {}
prod_tables_changed = []

for table_name in PROD_TABLES:
    if spark.catalog.tableExists(table_name):
        prod_versions_after[table_name] = spark.sql(
            f"DESCRIBE HISTORY {table_name} LIMIT 1"
        ).collect()[0]["version"]
    else:
        prod_versions_after[table_name] = -1

    print(table_name, ":", prod_versions_before[table_name], "->", prod_versions_after[table_name])

    if prod_versions_after[table_name] != prod_versions_before[table_name]:
        prod_tables_changed.append(table_name)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit rows of the three stopped runs' AS check_name,
               CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'bad.run.id')) AS STRING) AS actual,
               '0' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 runs of the final audit by status',
               concat('SUCCEEDED=', CAST(COUNT(DISTINCT CASE WHEN status = 'SUCCEEDED' THEN run_id END) AS STRING),
                      ' FAILED=', CAST(COUNT(DISTINCT CASE WHEN status = 'FAILED' THEN run_id END) AS STRING),
                      ' NO_INPUT=', CAST(COUNT(DISTINCT CASE WHEN status = 'NO_INPUT' THEN run_id END) AS STRING)),
               'SUCCEEDED=4 FAILED=3 NO_INPUT=2'
        FROM {TEST_AUDIT} WHERE pipeline_name = 'gold_final_audit'
        UNION ALL
        SELECT '03 audit rows of the 6 other runs of the final audit (one row each)',
               CAST(count_if(run_id NOT IN ('test-audit-run1', 'test-audit-run5', 'test-audit-run7')) AS STRING), '6'
        FROM {TEST_AUDIT} WHERE pipeline_name = 'gold_final_audit'
        UNION ALL
        SELECT '04 audit rows of runs 1, 5 and 7 that are not FAILED',
               CAST(count_if(run_id IN ('test-audit-run1', 'test-audit-run5', 'test-audit-run7') AND status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT} WHERE pipeline_name = 'gold_final_audit'
        UNION ALL
        SELECT '05 final audit rows with a Gold table, Gold rows or a table version',
               CAST(count_if(gold_table IS NOT NULL OR gold_rows_published <> 0 OR gold_table_version IS NOT NULL) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE pipeline_name = 'gold_final_audit'
        UNION ALL
        SELECT '06 runs of the six Gold notebooks by status',
               concat('SUCCEEDED=', CAST(count_if(status = 'SUCCEEDED') AS STRING),
                      ' FAILED=', CAST(count_if(status = 'FAILED') AS STRING),
                      ' NO_INPUT=', CAST(count_if(status = 'NO_INPUT') AS STRING)),
               'SUCCEEDED=9 FAILED=0 NO_INPUT=6'
        FROM {TEST_AUDIT} WHERE pipeline_name <> 'gold_final_audit'
        UNION ALL
        SELECT '07 production tables that changed', '{len(prod_tables_changed)}', '0'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T26"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T26: {failed_count} check(s) failed. See the table above.")
print("T26: ALL CHECKS PASSED")

## Result of the whole test

Every section wrote its number of failed checks into `section_results`. This cell goes through the 13 sections. A section that is not in the dictionary did not run, for example because the cells above it were skipped.

The test has passed only when this cell prints `TEST PASSED`.

In [0]:
EXPECTED_SECTIONS = [
    "T4", "T8", "T10", "T12", "T14", "T16", "T18",
    "T20", "T22", "T23", "T24", "T25", "T26",
]

problems = []

for section in EXPECTED_SECTIONS:
    if section not in section_results:
        problems.append(section + " did not run")
    elif section_results[section] > 0:
        problems.append(section + " has " + str(section_results[section]) + " failed check(s)")

print("Sections that ran:", len(section_results), "of", len(EXPECTED_SECTIONS))

if len(problems) > 0:
    raise AssertionError("TEST FAILED: " + "; ".join(problems))

print("TEST PASSED: all", len(EXPECTED_SECTIONS), "sections ran and every check passed.")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

If the test failed, you can leave this cell out and look at the test tables first. The next test run drops the schema anyway, in T1.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)